# Laboratorio 3: Predicción de la Progresión de la Diabetes con Regresión Lineal

**Objetivo:** Construir un modelo de regresión lineal que prediga la progresión de la diabetes un año después de la evaluación inicial, utilizando el dataset `Diabetes` de scikit-learn.

**Flujo del laboratorio:**
1. Carga y exploración del dataset
2. Análisis exploratorio (EDA)
3. Preprocesamiento (split + escalado)
4. Entrenamiento del modelo
5. Evaluación con MAE, MSE, RMSE y R²
6. Visualización de resultados e interpretación

## 1. Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print('Librerías importadas correctamente ✓')

## 2. Carga y exploración del dataset

In [ ]:
diabetes = load_diabetes()

X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y = pd.Series(diabetes.target, name='Progresión')

print('Descripción del dataset:')
print(diabetes.DESCR[:900])

In [ ]:
print(f'Forma del dataset: {X.shape}  →  {X.shape[0]} pacientes, {X.shape[1]} características')
print(f'Variable objetivo: "{y.name}" — rango [{y.min():.0f}, {y.max():.0f}]')
print()
print('Características disponibles:')
for i, col in enumerate(X.columns, 1):
    print(f'  {i:2}. {col}')

In [ ]:
print('Primeras 5 filas del dataset:')
X.head()

In [ ]:
print('Estadísticas descriptivas:')
X.describe().round(4)

In [ ]:
print('Valores nulos por columna:')
print(X.isnull().sum())
print('\nNo hay valores faltantes ✓' if X.isnull().sum().sum() == 0 else '⚠️ Hay valores faltantes')

## 3. Análisis Exploratorio (EDA)

In [ ]:
# Distribución de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(y, bins=25, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de la variable objetivo\n(Progresión de diabetes)', fontsize=12)
axes[0].set_xlabel('Progresión (un año después)')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(y.mean(), color='red', linestyle='--', label=f'Media: {y.mean():.1f}')
axes[0].axvline(y.median(), color='orange', linestyle='--', label=f'Mediana: {y.median():.1f}')
axes[0].legend()

# Boxplot
axes[1].boxplot(y, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Boxplot – Variable objetivo', fontsize=12)
axes[1].set_ylabel('Progresión')
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

print(f'Media: {y.mean():.2f}  |  Mediana: {y.median():.2f}  |  Std: {y.std():.2f}')

In [ ]:
# Mapa de correlación
df_full = X.copy()
df_full['Progresión'] = y

plt.figure(figsize=(12, 8))
corr = df_full.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, vmin=-1, vmax=1, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Mapa de Correlación – Dataset Diabetes', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Correlación de cada variable con el target
correlaciones = df_full.corr()['Progresión'].drop('Progresión').sort_values(ascending=False)

plt.figure(figsize=(9, 4))
colores = ['steelblue' if v > 0 else 'tomato' for v in correlaciones]
correlaciones.plot(kind='bar', color=colores, edgecolor='white')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Correlación de cada característica con la Progresión', fontsize=12)
plt.ylabel('Correlación de Pearson')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Variables con mayor correlación positiva con la progresión:')
print(correlaciones.head(3))

## 4. Preprocesamiento

### 4.1 División en conjuntos de entrenamiento y prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Conjunto de entrenamiento: {X_train.shape[0]} muestras ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Conjunto de prueba:        {X_test.shape[0]} muestras ({X_test.shape[0]/len(X)*100:.0f}%)')

### 4.2 Escalado con StandardScaler

Aunque el dataset de diabetes ya viene pre-normalizado, aplicar `StandardScaler` es buena práctica y refuerza el flujo correcto de trabajo.

In [ ]:
scaler = StandardScaler()

# fit SOLO en train, transform en ambos
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Escalado aplicado correctamente ✓')
print(f'Media en entrenamiento escalado (≈0): {X_train_sc.mean():.6f}')
print(f'Std  en entrenamiento escalado (≈1): {X_train_sc.std():.6f}')

## 5. Entrenamiento del Modelo de Regresión Lineal

In [ ]:
model = LinearRegression()
model.fit(X_train_sc, y_train)

print('Modelo entrenado ✓')
print(f'Intercepto (β₀): {model.intercept_:.4f}')
print()
print('Coeficientes por característica:')
coeficientes = pd.Series(model.coef_, index=X.columns).sort_values(ascending=False)
for nombre, coef in coeficientes.items():
    signo = '+' if coef > 0 else ''
    print(f'  {nombre:>6}: {signo}{coef:.4f}')

In [ ]:
# Visualizar coeficientes
plt.figure(figsize=(9, 4))
colores = ['steelblue' if v > 0 else 'tomato' for v in coeficientes]
coeficientes.plot(kind='bar', color=colores, edgecolor='white')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Coeficientes del modelo de Regresión Lineal', fontsize=12)
plt.ylabel('Valor del coeficiente')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Predicciones

In [ ]:
y_pred = model.predict(X_test_sc)

comparacion = pd.DataFrame({
    'Real':     y_test.values,
    'Predicho': y_pred.round(2),
    'Error':    (y_test.values - y_pred).round(2)
})

print('Primeras 10 predicciones vs valores reales:')
comparacion.head(10)

## 7. Evaluación del Modelo

| Métrica | Descripción |
|---------|-------------|
| **MAE** | Error Absoluto Medio — cuánto se equivoca en promedio (misma escala que y) |
| **MSE** | Error Cuadrático Medio — penaliza errores grandes |
| **RMSE** | Raíz del MSE — interpretable en la escala original de y |
| **R²** | Proporción de varianza explicada (0 = pésimo, 1 = perfecto) |

In [ ]:
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print('=' * 40)
print('   EVALUACIÓN FINAL DEL MODELO')
print('=' * 40)
print(f'  MAE  (Error Absoluto Medio) : {mae:.4f}')
print(f'  MSE  (Error Cuadrático Med) : {mse:.4f}')
print(f'  RMSE (Raíz del MSE)         : {rmse:.4f}')
print(f'  R²   (Coef. determinación)  : {r2:.4f}')
print('=' * 40)
print(f'\nInterpretación: el modelo explica el {r2*100:.1f}% de la')
print(f'variabilidad en la progresión de la diabetes.')
print(f'En promedio, se equivoca {mae:.1f} unidades (MAE).')

## 8. Visualización de Resultados

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Predicho vs Real ---
axes[0].scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='white', s=55)
lims = [min(y_test.min(), y_pred.min()) - 10, max(y_test.max(), y_pred.max()) + 10]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Predicción perfecta')
axes[0].set_xlabel('Valor Real')
axes[0].set_ylabel('Valor Predicho')
axes[0].set_title('Predicho vs Real', fontsize=12)
axes[0].legend()

# --- Residuales ---
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.6, color='seagreen', edgecolors='white', s=55)
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Valor Predicho')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residuales vs Predichos', fontsize=12)

# --- Distribución de residuales ---
axes[2].hist(residuals, bins=20, color='darkorange', edgecolor='white')
axes[2].axvline(0, color='red', linestyle='--', linewidth=2)
axes[2].set_xlabel('Residual')
axes[2].set_ylabel('Frecuencia')
axes[2].set_title('Distribución de Residuales', fontsize=12)

plt.suptitle('Evaluación del Modelo – Regresión Lineal (Diabetes)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 9. Preguntas de reflexión

**Pregunta 1:** ¿Qué significa el valor de R² obtenido?

> **Respuesta:** El R² de **0.4526** significa que el modelo explica aproximadamente el **45.3% de la variabilidad** en la progresión de la diabetes. Es decir, cerca de la mitad de las diferencias entre pacientes queda explicada por las 10 características médicas del dataset. El 54.7% restante se debe a factores no incluidos en el modelo (hábitos de vida, genética, medicación, etc.). Un R² de ~0.45 es razonable en datos médicos reales, donde la variabilidad biológica es alta.

**Pregunta 2:** ¿Qué características tienen mayor impacto en la predicción según los coeficientes?

> **Respuesta:** Las variables `bmi` (Índice de Masa Corporal) y `s5` (nivel de triglicéridos en suero) presentan los coeficientes positivos más altos, lo que indica que un aumento en estas variables está asociado con una mayor progresión de la diabetes. Esto es consistente con el conocimiento médico: el sobrepeso y los lípidos elevados son factores de riesgo bien documentados para la diabetes tipo 2.

**Pregunta 3:** ¿Por qué es importante que los residuales tengan distribución aproximadamente normal?

> **Respuesta:** La normalidad de los residuales es uno de los supuestos de la regresión lineal clásica (OLS). Si se cumple, los estimadores de los coeficientes son los mejores estimadores lineales insesgados (Teorema de Gauss-Markov) y los intervalos de confianza e hipótesis estadísticas son válidos. Si los residuales tienen sesgo o colas muy pesadas, puede indicar que el modelo lineal no captura adecuadamente la relación subyacente y habría que considerar transformaciones o modelos más complejos.

**Pregunta 4:** ¿Qué mejoras podrían incrementar el R²?

> **Respuesta:** Algunas estrategias para mejorar el modelo incluyen: (1) **Ingeniería de features**: crear variables derivadas o interacciones entre características (ej. `bmi × s5`). (2) **Modelos más complejos**: Ridge, Lasso o Elastic Net para regularización, o modelos no lineales como Random Forest o Gradient Boosting. (3) **Selección de características**: eliminar variables con baja correlación con el target. (4) **Más datos**: mayor cantidad de pacientes reduciría el ruido en las estimaciones.

## 10. Resumen del pipeline completo

In [ ]:
from sklearn.pipeline import Pipeline

# Pipeline compacto: escalado + modelo en un solo objeto
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LinearRegression())
])

pipeline.fit(X_train, y_train)
y_pred_pipe = pipeline.predict(X_test)

r2_pipe = r2_score(y_test, y_pred_pipe)
print(f'R² usando Pipeline: {r2_pipe:.4f}  (idéntico al manual ✓)')
print()
print('El Pipeline garantiza que el escalado se aplique correctamente')
print('tanto en entrenamiento como en producción.')

---
## Conclusión

En este laboratorio construimos un modelo de **regresión lineal** completo para predecir la progresión de la diabetes:

- Se cargó y exploró el dataset con 442 pacientes y 10 características médicas
- Se realizó EDA con mapa de correlación y distribuciones
- Se dividió el dataset 80/20 y se aplicó `StandardScaler`
- El modelo obtuvo un **R² = 0.4526**, explicando ~45% de la variabilidad
- El **RMSE = 53.85** indica que las predicciones se desvían ~54 unidades en promedio
- Los residuales mostraron distribución aproximadamente normal, validando los supuestos del modelo
- Se implementó el modelo como un `Pipeline` de sklearn para su uso en producción